# D02 — External Biological Databases

Downloads and caches the external reference databases used for biological
annotation in the reported analyses.

**Outputs (in `data/external/`):**
- `gnomad.v4.1.constraint_metrics.tsv` — gnomAD v4.1 gene constraint (~92 MB)
- `clinvar_gene_specific_summary.txt` — ClinVar gene-level pathogenicity (~3.4 MB)

**Outputs (in `data/`):**
- `hpa_rna_consensus.tsv.zip` — Human Protein Atlas tissue expression (~5 MB)

**Prerequisites:** `requests`, `pandas`

## Section 1: gnomAD v4.1 Constraint Metrics

gnomAD provides population-level constraint metrics. pLI (probability of loss-of-function intolerance) and LOEUF (loss-of-function observed/expected upper fraction) quantify selective constraint against gene inactivation. Highly constrained genes (pLI > 0.9) are more likely to be essential.

In [ ]:
import requests
from pathlib import Path
import os

DATA_DIR = Path('data/external')
DATA_DIR.mkdir(parents=True, exist_ok=True)

GNOMAD_URL = 'https://storage.googleapis.com/gcp-public-data--gnomad/release/4.1/constraint/gnomad.v4.1.constraint_metrics.tsv'
GNOMAD_PATH = DATA_DIR / 'gnomad.v4.1.constraint_metrics.tsv'

if GNOMAD_PATH.exists():
    size_mb = GNOMAD_PATH.stat().st_size / 1e6
    print(f'gnomAD constraint metrics already cached ({size_mb:.1f} MB)')
else:
    print(f'Downloading gnomAD v4.1 constraint metrics...')
    print(f'  Source: {GNOMAD_URL}')
    resp = requests.get(GNOMAD_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    
    downloaded = 0
    with open(GNOMAD_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.0f}%)', end='', flush=True)
    print(f'\n  Saved to {GNOMAD_PATH}')

In [ ]:
import pandas as pd

gnomad = pd.read_csv(GNOMAD_PATH, sep='\t', nrows=5)
print(f'Columns ({len(gnomad.columns)}): {list(gnomad.columns[:10])}...')

# Column names vary by gnomAD version — find the right ones
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

gnomad_full = pd.read_csv(GNOMAD_PATH, sep='\t', low_memory=False)
gene_col  = find_col(gnomad_full, ['gene', 'gene_id', 'symbol'])
pli_col   = find_col(gnomad_full, ['lof.pLI', 'pLI'])
loeuf_col = find_col(gnomad_full, ['lof.oe_ci.upper', 'oe_lof_upper', 'LOEUF'])

print(f'Loaded {len(gnomad_full):,} rows')
print(f'Gene column: {gene_col} ({gnomad_full[gene_col].nunique():,} unique genes)')
print(f'pLI column: {pli_col}')
print(f'LOEUF column: {loeuf_col}')


## Section 2: ClinVar Gene-Specific Summary

ClinVar aggregates genetic variant interpretations. The gene_specific_summary file lists per-gene counts of pathogenic, likely pathogenic, and benign variants, enabling classification of genes as "disease-associated."

In [ ]:
CLINVAR_URL = 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/gene_specific_summary.txt'
CLINVAR_PATH = DATA_DIR / 'clinvar_gene_specific_summary.txt'

if CLINVAR_PATH.exists():
    size_mb = CLINVAR_PATH.stat().st_size / 1e6
    print(f'ClinVar gene-specific summary already cached ({size_mb:.1f} MB)')
else:
    print(f'Downloading ClinVar gene-specific summary...')
    print(f'  Source: {CLINVAR_URL}')
    resp = requests.get(CLINVAR_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    
    downloaded = 0
    with open(CLINVAR_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.0f}%)', end='', flush=True)
    print(f'\n  Saved to {CLINVAR_PATH}')

In [ ]:
# ClinVar gene-specific summary has a comment header line starting with '#'.
# Read first line to decide whether to skip it (robust to format changes).
with open(CLINVAR_PATH) as _f:
    _first = _f.readline()
_skip = 1 if _first.startswith('#') else 0
clinvar = pd.read_csv(CLINVAR_PATH, sep='\t', skiprows=_skip, low_memory=False)
# Strip any leading '#' from column names (defensive)
clinvar.columns = [c.lstrip('#').strip() for c in clinvar.columns]
print(f'ClinVar gene-specific summary: {clinvar.shape}')
print(f'Columns: {list(clinvar.columns)}')

## Section 3: Human Protein Atlas (HPA) RNA Consensus

HPA provides consensus tissue expression levels (nTPM) across ~60 human tissues. Used to compute expression breadth (number of tissues where a gene is expressed above 1 TPM) and maximum tissue expression.

In [ ]:
import zipfile

HPA_URL = 'https://www.proteinatlas.org/download/tsv/rna_tissue_consensus.tsv.zip'
HPA_PATH = Path('data/hpa_rna_consensus.tsv.zip')
HPA_PATH.parent.mkdir(parents=True, exist_ok=True)

if HPA_PATH.exists():
    size_mb = HPA_PATH.stat().st_size / 1e6
    print(f'HPA RNA consensus already cached ({size_mb:.1f} MB)')
else:
    print(f'Downloading Human Protein Atlas RNA consensus...')
    print(f'  Source: {HPA_URL}')
    resp = requests.get(HPA_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    
    downloaded = 0
    with open(HPA_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.0f}%)', end='', flush=True)
    print(f'\n  Saved to {HPA_PATH}')

In [ ]:
with zipfile.ZipFile(HPA_PATH, 'r') as zf:
    file_list = zf.namelist()
    print(f'ZIP contents: {file_list}')
    with zf.open(file_list[0]) as f:
        hpa = pd.read_csv(f, sep='\t', nrows=1000)
        print(f'HPA RNA consensus: {hpa.shape}')
        print(f'First 5 columns: {list(hpa.columns[:5])}')

## Section 4: Verification Summary

Verify all downloaded files exist and report their sizes.

In [ ]:
print('External Database Files:')
print('=' * 60)

files_to_check = [
    (GNOMAD_PATH, 'gnomAD v4.1 constraint metrics'),
    (CLINVAR_PATH, 'ClinVar gene-specific summary'),
    (HPA_PATH, 'HPA RNA consensus (zipped)')
]

total_size = 0
for path, description in files_to_check:
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        total_size += path.stat().st_size
        status = '✓'
    else:
        size_mb = 0
        status = '✗'
    print(f'{status} {description:.<45} {size_mb:>8.1f} MB')

print('=' * 60)
print(f'Total size: {total_size / 1e9:.2f} GB')